# HABIT v1 habitat API quickstart

Minimal in-memory workflow: synthetic cohort → two-step habitat analysis → save artefacts.

This notebook uses the v1 Python API only (no CLI, no YAML).

In [ ]:
from habit import HabitatSpec, Spec, check_component, make_synthetic_cohort, show_versions
import habit.recipes as recipes

# Record the software stack for reproducibility
for package, version in show_versions().items():
    print(f"{package}: {version}")

In [ ]:
# Build a small deterministic cohort entirely in memory (no demo_data required)
cohort = make_synthetic_cohort(
    n_subjects=2,
    modalities=("T1", "T2"),
    shape=(24, 24, 24),
    rng=42,
)
print(f"Cohort: {cohort.name}, subjects: {len(cohort)}")

In [ ]:
# Pre-flight: confirm the declared components resolve through the registry
assert check_component("raw", domain="voxel_feature_extractor")
assert check_component("slic", domain="supervoxelizer")
assert check_component("kmeans", domain="habitat_model_fitter")
assert check_component("nearest_centroid", domain="habitat_assigner")

spec = HabitatSpec(
    name="quickstart_two_step",
    voxel_feature_extractor=Spec("raw", {"modalities": ["T1", "T2"]}),
    supervoxelizer=Spec("slic", {"n_supervoxels": 12}),
    habitat_model_fitter=Spec("kmeans", {"n_habitats": 3, "n_init": 3}),
    habitat_assigner=Spec("nearest_centroid"),
    habitat_features=(Spec("volume"),),
    random_seed=42,
)

result = recipes.two_step(cohort, spec)
print(result.habitat_model.summary())

In [ ]:
import tempfile
from pathlib import Path

# Persist to a temporary directory (explicit opt-in, v0.1-compatible layout)
out_dir = Path(tempfile.mkdtemp(prefix="habit_quickstart_"))
saved = result.save(out_dir)
print(f"Saved to: {saved}")
print("Files:", sorted(p.name for p in saved.iterdir()))